# Day 3 — LLM Quantisation
### FP16 → INT8 → INT4 → GGUF

**Flow:**
```
Day 2 Adapters
     │
     ▼
Merge → FP16 (baseline)
     │
     ├──► INT8  (bitsandbytes, GPU)
     ├──► INT4  (bitsandbytes, GPU)
     └──► GGUF  (llama.cpp, CPU)
```


In [18]:
# GPU Check
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU! Go to: Runtime > Change runtime type > T4 GPU')

GPU: Tesla T4
VRAM: 15.6 GB


In [19]:
# Install dependencies
!pip install transformers peft bitsandbytes accelerate -q
print('Core packages installed!')

Core packages installed!


In [20]:
#Setup output folders
import os

BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_PATH = '/content/adapters'

OUTPUT_DIR = '/content/quantized'
MERGED_DIR = f'{OUTPUT_DIR}/merged_fp16'
INT8_DIR   = f'{OUTPUT_DIR}/model-int8'
INT4_DIR   = f'{OUTPUT_DIR}/model-int4'
GGUF_PATH  = f'{OUTPUT_DIR}/model.gguf'

for d in [OUTPUT_DIR, MERGED_DIR, INT8_DIR, INT4_DIR]:
    os.makedirs(d, exist_ok=True)

print('Output folders created:')
print(f'   {OUTPUT_DIR}/')
print(f'   ├── merged_fp16/')
print(f'   ├── model-int8/')
print(f'   ├── model-int4/')
print(f'   └── model.gguf')

Output folders created:
   /content/quantized/
   ├── merged_fp16/
   ├── model-int8/
   ├── model-int4/
   └── model.gguf


In [21]:
#Load base model + merge LoRA adapters → save FP16
# Why merge first?
# bitsandbytes quantises a SINGLE merged model, not base + adapter separately.
# Merging folds adapter weights back into the base weights permanently.
from transformers import AutoModelForCausalLM, AutoTokenizer
import time, gc

def get_vram_gb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e9
    return 0.0

def get_dir_size_gb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1e9

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'Loading tokenizer: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print('Loading FP16 model...')
clear_gpu()
vram_before = get_vram_gb()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)

# Load LoRA adapter
if ADAPTER_PATH and os.path.exists(ADAPTER_PATH):
    from peft import PeftModel
    print(f'Loading LoRA adapter from {ADAPTER_PATH}...')
    model_fp16 = PeftModel.from_pretrained(model_fp16, ADAPTER_PATH)
    print('Merging adapter into base model...')
    model_fp16 = model_fp16.merge_and_unload()  # Bake adapter into weights
    print('Adapter merged!')
else:
    print('[INFO] No adapter found. Using base model (add Day 2 adapter path for full pipeline)')

vram_fp16 = get_vram_gb() - vram_before
print(f'\nFP16 model loaded!')
print(f'   VRAM used: {vram_fp16:.2f} GB')
print(f'   Params: {sum(p.numel() for p in model_fp16.parameters()):,}')

Loading tokenizer: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Loading FP16 model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading LoRA adapter from /content/adapters...
Merging adapter into base model...
Adapter merged!

FP16 model loaded!
   VRAM used: 2.20 GB
   Params: 1,100,048,384


In [22]:
#Measure FP16 inference speed
TEST_PROMPT = '### Instruction:\nWhat is a list in Python?\n\n### Response:\n'

def measure_speed(model, tokenizer, prompt, n_tokens=50, label=''):
    inputs = tokenizer(prompt, return_tensors='pt')
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    # Warmup
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)

    # Actual timing
    if torch.cuda.is_available(): torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=n_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    if torch.cuda.is_available(): torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    new_tokens = out.shape[1] - inputs['input_ids'].shape[1]
    tps = new_tokens / elapsed
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    print(f'\n[{label}]')
    print(f'  Speed:    {tps:.1f} tokens/sec')
    print(f'  Time:     {elapsed:.2f}s for {new_tokens} tokens')
    print(f'  Response: {response[:150]}...')
    return round(tps, 1)

fp16_tps = measure_speed(model_fp16, tokenizer, TEST_PROMPT, label='FP16')

# Save FP16 model
print(f'\nSaving FP16 merged model to {MERGED_DIR}...')
model_fp16.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
fp16_size = get_dir_size_gb(MERGED_DIR)
print(f'Saved! Size: {fp16_size:.2f} GB')

# Free memory
del model_fp16
clear_gpu()
print('GPU memory freed')


[FP16]
  Speed:    6.6 tokens/sec
  Time:     3.04s for 20 tokens
  Response: A list is an ordered and mutable data structure that can store multiple items of different data types....

Saving FP16 merged model to /content/quantized/merged_fp16...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved! Size: 2.20 GB
GPU memory freed


## How INT8 works:
####   FP16 weights → scale factor computed per row/column → stored as int8
####   During forward pass: dequantise on-the-fly to compute in FP16
####   Result: ~50% memory reduction, <1% accuracy loss on most tasks

In [23]:
from transformers import BitsAndBytesConfig

print('Loading model in INT8...')
print('(2x smaller than FP16, almost same quality)')

clear_gpu()
vram_before = get_vram_gb()

bnb_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,  # Outlier threshold: weights > 6.0 stay in FP16
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,              # Load from our saved FP16 merged model
    quantization_config=bnb_8bit,
    device_map='auto',
)

vram_int8 = get_vram_gb() - vram_before
print(f'\nINT8 model loaded!')
print(f'   VRAM used: {vram_int8:.2f} GB')
print(f'   vs FP16:   {vram_fp16:.2f} GB')
if vram_fp16 > 0:
    print(f'   Savings:   {((vram_fp16 - vram_int8)/vram_fp16*100):.0f}% less VRAM')
else:
    print('   Savings:   Cannot calculate VRAM savings without GPU data (vram_fp16 is 0)')

int8_tps = measure_speed(model_int8, tokenizer, TEST_PROMPT, label='INT8')

# Save INT8 model and info
print(f'\nSaving INT8 model and config to {INT8_DIR}...')
model_int8.save_pretrained(INT8_DIR) # Save the model itself
tokenizer.save_pretrained(INT8_DIR)
int8_disk_size = get_dir_size_gb(INT8_DIR) # Measure disk size after saving
print(f'INT8 model saved! Disk size: {int8_disk_size:.2f} GB')

# Update README with actual disk size for clarity
with open(f'{INT8_DIR}/README.md', 'w') as f:
    f.write(f'# INT8 Quantized Model\n\n')
    f.write(f'Base: {BASE_MODEL}\n')
    f.write(f'VRAM: ~{vram_int8:.2f} GB (In-memory)\n')
    f.write(f'Disk Size: ~{int8_disk_size:.2f} GB (Weights saved mostly as FP16 with quantization config)\n')
    f.write(f'Speed: {int8_tps} tok/s\n\n')
    f.write('## Load code:\n```python\n')
    f.write('from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n')
    f.write('bnb = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)\n')
    f.write(f'model = AutoModelForCausalLM.from_pretrained("'+INT8_DIR+'", quantization_config=bnb)\n```') # Load from saved path
print(f'INT8 config saved to {INT8_DIR}')

del model_int8
clear_gpu()
print('GPU memory freed')


Loading model in INT8...
(2x smaller than FP16, almost same quality)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


INT8 model loaded!
   VRAM used: 1.24 GB
   vs FP16:   2.20 GB
   Savings:   44% less VRAM

[INT8]
  Speed:    7.3 tokens/sec
  Time:     2.73s for 20 tokens
  Response: A list is an ordered and mutable data structure that can store multiple items of different data types....

Saving INT8 model and config to /content/quantized/model-int8...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 model saved! Disk size: 1.24 GB
INT8 config saved to /content/quantized/model-int8
GPU memory freed


## How NF4 works:
####   NF4 = Normal Float 4-bit: quantisation levels spaced to match
####   a normal distribution (which neural network weights follow)
####   Double quantisation: quantises the scale factors themselves too
####   Result: ~75% memory reduction vs FP16

In [ ]:
# INT4 / NF4 Quantization
print('Loading model in INT4 (NF4)...')
print('(4x smaller than FP16!)')

clear_gpu()
vram_before = get_vram_gb()

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',           # Normal Float 4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.float16, # Compute in FP16 (not FP32)
    bnb_4bit_use_double_quant=True,       # Quantize the quantization constants too
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    quantization_config=bnb_4bit,
    device_map='auto',
)

vram_int4 = get_vram_gb() - vram_before
print(f'\nINT4 model loaded!')
print(f'   VRAM used: {vram_int4:.2f} GB')
print(f'   vs FP16:   {vram_fp16:.2f} GB')
if vram_fp16 > 0:
    print(f'   Savings:   {((vram_fp16 - vram_int4)/vram_fp16*100):.0f}% less VRAM')
else:
    print('   Savings:   Cannot calculate VRAM savings without GPU data (vram_fp16 is 0)')

int4_tps = measure_speed(model_int4, tokenizer, TEST_PROMPT, label='INT4 NF4')

# Save INT4 model and info
print(f'\nSaving INT4 model and config to {INT4_DIR}...')
model_int4.save_pretrained(INT4_DIR) # Save the model itself
tokenizer.save_pretrained(INT4_DIR)
int4_disk_size = get_dir_size_gb(INT4_DIR) # Measure disk size after saving
print(f'INT4 model saved! Disk size: {int4_disk_size:.2f} GB')

# Update README with actual disk size for clarity
with open(f'{INT4_DIR}/README.md', 'w') as f:
    f.write(f'# INT4 (NF4) Quantized Model\n\n')
    f.write(f'Base: {BASE_MODEL}\n')
    f.write(f'VRAM: ~{vram_int4:.2f} GB (In-memory)\n')
    f.write(f'Disk Size: ~{int4_disk_size:.2f} GB (Weights saved mostly as FP16 with quantization config)\n')
    f.write(f'Speed: {int4_tps} tok/s\n\n')
    f.write('## Load code:\n```python\n')
    f.write('from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n')
    f.write('bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n')
    f.write('                         bnb_4bit_compute_dtype=torch.float16)\n')
    f.write(f'model = AutoModelForCausalLM.from_pretrained("'+INT4_DIR+'", quantization_config=bnb)\n```') # Load from saved path
print(f' INT4 config saved to {INT4_DIR}')

del model_int4
clear_gpu()
print('GPU memory freed ✅')


Loading model in INT4 (NF4)...
(4x smaller than FP16!)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


INT4 model loaded!
   VRAM used: 0.77 GB
   vs FP16:   2.20 GB
   Savings:   65% less VRAM

[INT4 NF4]
  Speed:    18.7 tokens/sec
  Time:     1.07s for 20 tokens
  Response: A list is an ordered and mutable data structure that can store multiple items of different data types....

Saving INT4 model and config to /content/quantized/model-int4...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 model saved! Disk size: 0.77 GB
 INT4 config saved to /content/quantized/model-int4
GPU memory freed ✅


## llama.cpp: C++ inference engine that reads GGUF format.
#### GGUF = GPT-Generated Unified Format (replaces old GGML).
#### Q4_0 = 4-bit quantisation, group size 0 (simplest, fastest, smallest)
#### Q8_0 = 8-bit quantisation (larger but better quality)

In [25]:
#  Install llama.cpp
import subprocess, sys

LLAMA_CPP_DIR = '/content/llama.cpp'

if not os.path.exists(LLAMA_CPP_DIR):
    print('Cloning llama.cpp...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/ggerganov/llama.cpp', LLAMA_CPP_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('llama.cpp cloned!')
    else:
        print(f'Clone failed: {result.stderr}')
else:
    print('llama.cpp already present')

# Install llama.cpp Python requirements
req = f'{LLAMA_CPP_DIR}/requirements.txt'
if os.path.exists(req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req, '-q'])
    print('llama.cpp requirements installed')

llama.cpp already present
llama.cpp requirements installed


In [26]:
print('Upgrading tokenizers package to fix potential backend issues...')
!pip install --upgrade tokenizers -q
print('tokenizers upgraded!')


Upgrading tokenizers package to fix potential backend issues...
tokenizers upgraded!


In [27]:
print('Reinstalling specific versions of transformers and tokenizers for compatibility...')
!pip install transformers==4.32.1 tokenizers==0.13.3 -q
print('transformers and tokenizers reinstalled!')

Reinstalling specific versions of transformers and tokenizers for compatibility...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
transformers and tokenizers reinstalled!


In [28]:
import sys
import os

print('Python sys.path:')
for p in sys.path:
    print(p)

print('\nTrying to import LlamaConfig explicitly...')
try:
    from transformers import LlamaConfig
    print('LlamaConfig imported successfully from transformers!')
except ImportError as e:
    print(f'Failed to import LlamaConfig: {e}')
    print('This indicates a problem with the transformers installation or environment path.')


Python sys.path:
/content
/env/python
/usr/lib/python312.zip
/usr/lib/python3.12
/usr/lib/python3.12/lib-dynload

/usr/local/lib/python3.12/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.12/dist-packages/IPython/extensions
/root/.ipython

Trying to import LlamaConfig explicitly...
LlamaConfig imported successfully from transformers!


In [29]:
# Convert HuggingFace -> GGUF
# Step 1: FP16 GGUF (intermediate format)

TEMP_GGUF = f'{OUTPUT_DIR}/model_fp16.gguf'
QUANT_TYPE = 'q4_0'   # Change to 'q8_0' for better quality

# Find convert script
convert_scripts = [
    f'{LLAMA_CPP_DIR}/convert_hf_to_gguf.py',
    f'{LLAMA_CPP_DIR}/convert.py',
]
convert_script = None
for s in convert_scripts:
    if os.path.exists(s):
        convert_script = s
        break

if not convert_script:
    print(' Conversion script not found in llama.cpp')
    print('   Try: !ls /content/llama.cpp/*.py')
else:
    print(f'Using conversion script: {convert_script}')
    print(f'Converting {MERGED_DIR} → {TEMP_GGUF}...')
    print('(This takes 2-5 minutes)')

    # --- NEW: Ensure tokenizer.model is present for llama.cpp conversion ---
    from huggingface_hub import snapshot_download
    import shutil

    print('Ensuring tokenizer.model is available for GGUF conversion...')
    # Download the model files to a local cache, if not already present
    # This ensures tokenizer.model is downloaded if it exists in the repo
    model_cache_path = snapshot_download(repo_id=BASE_MODEL)

    # Source path for tokenizer.model (it exists in TinyLlama repo)
    src_tokenizer_model_path = os.path.join(model_cache_path, 'tokenizer.model')

    # Destination path in our merged directory
    dst_tokenizer_model_path = os.path.join(MERGED_DIR, 'tokenizer.model')

    if os.path.exists(src_tokenizer_model_path):
        if not os.path.exists(dst_tokenizer_model_path) or os.path.getsize(src_tokenizer_model_path) != os.path.getsize(dst_tokenizer_model_path):
            print(f'Copying tokenizer.model from {src_tokenizer_model_path} to {dst_tokenizer_model_path}')
            shutil.copy(src_tokenizer_model_path, dst_tokenizer_model_path)
            print('tokenizer.model copied!')
        else:
            print('tokenizer.model already exists and is up-to-date in MERGED_DIR.')
    else:
        print(f'tokenizer.model not found at {src_tokenizer_model_path}. GGUF conversion might fail.')
    # --- END NEW ---

    # Capture current environment to pass to subprocess
    current_env = os.environ.copy()

    try:
        # Removed capture_output=True and text=True to stream output directly
        result = subprocess.run(
            [sys.executable, convert_script,
             MERGED_DIR,
             '--outfile', TEMP_GGUF,
             '--outtype', 'f16',
             '--verbose'],
            timeout=600, check=True,
            env=current_env # Pass the current environment
        )

        if result.returncode == 0:
            size = os.path.getsize(TEMP_GGUF) / 1e9
            print(f'FP16 GGUF created! Size: {size:.2f} GB')
    except subprocess.TimeoutExpired as e:
        print(f'Conversion timed out after {e.timeout} seconds!')
        print('\nManual command:')
        print(f'!python {convert_script} {MERGED_DIR} --outfile {TEMP_GGUF} --outtype f16 --verbose')
    except subprocess.CalledProcessError as e:
        print(f'Conversion failed with CalledProcessError!')
        print('Return code:', e.returncode)
        print('\nManual command:')
        print(f'!python {convert_script} {MERGED_DIR} --outfile {TEMP_GGUF} --outtype f16 --verbose')
    except Exception as e:
        print(f'An unexpected error occurred during conversion: {e}')

Using conversion script: /content/llama.cpp/convert_hf_to_gguf.py
Converting /content/quantized/merged_fp16 → /content/quantized/model_fp16.gguf...
(This takes 2-5 minutes)
Ensuring tokenizer.model is available for GGUF conversion...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

tokenizer.model already exists and is up-to-date in MERGED_DIR.
FP16 GGUF created! Size: 2.20 GB


In [30]:
# Running the GGUF conversion command directly to get full error logs
!python /content/llama.cpp/convert_hf_to_gguf.py /content/quantized/merged_fp16 --outfile /content/quantized/model_fp16.gguf --outtype f16 --verbose

INFO:hf-to-gguf:Loading model: merged_fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.wei

In [31]:
# ─── CELL 10: Build llama-quantize & quantize GGUF ─────────────
# Step 2: Quantize the FP16 GGUF to q4_0

# Build llama.cpp (needed for quantize binary)
print('Building llama.cpp (for quantize binary)...')

build_result_make = subprocess.run(
    ['make', '-j', '4', 'llama-quantize'],
    cwd=LLAMA_CPP_DIR,
    capture_output=True, text=True
)

quantize_bin = f'{LLAMA_CPP_DIR}/llama-quantize'

if build_result_make.returncode == 0 and os.path.exists(quantize_bin):
    print('llama-quantize built with make!')
    # ... (rest of the quantization logic remains the same for successful make)
else:
    print('make build failed.')
    print('STDOUT:', build_result_make.stdout[-1000:] if build_result_make.stdout else 'No stdout')
    print('STDERR:', build_result_make.stderr[-1000:] if build_result_make.stderr else 'No stderr')
    print('Trying alternative: cmake')

    # Alternative: cmake build
    cmake_dir = f'{LLAMA_CPP_DIR}/build'
    os.makedirs(cmake_dir, exist_ok=True)

    print('Running cmake configuration...')
    cmake_config_result = subprocess.run(
        ['cmake', '..', '-DLLAMA_CUBLAS=OFF'],
        cwd=cmake_dir,
        capture_output=True, text=True, check=False # Don't check for failure yet, print output first
    )
    print('STDOUT:', cmake_config_result.stdout[-1000:] if cmake_config_result.stdout else 'No stdout')
    print('STDERR:', cmake_config_result.stderr[-1000:] if cmake_config_result.stderr else 'No stderr')

    if cmake_config_result.returncode != 0:
        print(' CMake configuration failed. Cannot proceed with build.')
    else:
        print('Running cmake build...')
        cmake_build_result = subprocess.run(
            ['cmake', '--build', '.', '--config', 'Release', '-j', '4'],
            cwd=cmake_dir,
            capture_output=True, text=True, check=False # Don't check for failure yet, print output first
        )
        print('STDOUT:', cmake_build_result.stdout[-1000:] if cmake_build_result.stdout else 'No stdout')
        print('STDERR:', cmake_build_result.stderr[-1000:] if cmake_build_result.stderr else 'No stderr')

        if cmake_build_result.returncode == 0:
            quantize_bin_alt = f'{cmake_dir}/bin/llama-quantize'
            if os.path.exists(quantize_bin_alt):
                print(' Built with cmake!')
                # Use the alternative quantize binary
                quantize_bin = quantize_bin_alt
            else:
                print(' CMake build succeeded, but llama-quantize binary not found.')
                print('[FALLBACK] Using FP16 GGUF without further quantization')
                # Move TEMP_GGUF to GGUF_PATH as fallback
                if os.path.exists(TEMP_GGUF):
                    shutil.move(TEMP_GGUF, GGUF_PATH)
                else:
                    print(f'Warning: {TEMP_GGUF} not found for fallback.')
                gguf_size = os.path.getsize(GGUF_PATH) / 1e9 if os.path.exists(GGUF_PATH) else 0
                print(f'GGUF size: {gguf_size:.2f} GB')
        else:
            print('CMake build failed.')
            print('[FALLBACK] Using FP16 GGUF without further quantization')
            # Move TEMP_GGUF to GGUF_PATH as fallback
            if os.path.exists(TEMP_GGUF):
                shutil.move(TEMP_GGUF, GGUF_PATH)
            else:
                print(f'Warning: {TEMP_GGUF} not found for fallback.')
            gguf_size = os.path.getsize(GGUF_PATH) / 1e9 if os.path.exists(GGUF_PATH) else 0
            print(f'GGUF size: {gguf_size:.2f} GB')

# This part only executes if a quantize_bin was successfully found/built
if os.path.exists(quantize_bin) and quantize_bin != f'{LLAMA_CPP_DIR}/llama-quantize': # Only if we successfully built or found a binary
    if os.path.exists(TEMP_GGUF):
        print(f'Quantizing {TEMP_GGUF} → {GGUF_PATH} ({QUANT_TYPE})...')
        q_result = subprocess.run(
            [quantize_bin, TEMP_GGUF, GGUF_PATH, QUANT_TYPE],
            capture_output=True, text=True, timeout=300, check=True
        )
        if q_result.returncode == 0:
            os.remove(TEMP_GGUF)  # Remove temp FP16 GGUF
            gguf_size = os.path.getsize(GGUF_PATH) / 1e9
            print(f' GGUF quantized! Size: {gguf_size:.2f} GB ({QUANT_TYPE})')
        else:
            print('Quantize failed:', q_result.stderr[-500:])
            # Keep FP16 GGUF as fallback
            import shutil
            shutil.move(TEMP_GGUF, GGUF_PATH) # Move it back to GGUF_PATH if quantization fails
            gguf_size = os.path.getsize(GGUF_PATH) / 1e9
            print(f'Using FP16 GGUF instead: {gguf_size:.2f} GB')
    else:
        print(f'Warning: {TEMP_GGUF} not found for quantization. Skipping quantization step.')
        if not os.path.exists(GGUF_PATH): # Ensure GGUF_PATH exists if TEMP_GGUF was already moved as fallback
            print('No GGUF file available after build attempts.')
else:
    print('Quantization binary not successfully built or found. Skipping quantization.')
    # If no quantize_bin is found after all attempts, ensure TEMP_GGUF (FP16 GGUF) is moved to GGUF_PATH if it exists
    if os.path.exists(TEMP_GGUF) and not os.path.exists(GGUF_PATH): # Only move if GGUF_PATH doesn't exist yet
        shutil.move(TEMP_GGUF, GGUF_PATH)
        gguf_size = os.path.getsize(GGUF_PATH) / 1e9
        print(f'Using FP16 GGUF as final output: {gguf_size:.2f} GB')
    elif os.path.exists(GGUF_PATH):
        gguf_size = os.path.getsize(GGUF_PATH) / 1e9
        print(f'GGUF size already established: {gguf_size:.2f} GB')
    else:
        print('No GGUF file available after build attempts or fallback.')



Building llama.cpp (for quantize binary)...
make build failed.
STDOUT: No stdout
STDERR: Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.

Trying alternative: cmake
Running cmake configuration...
STDOUT: -- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.11
-- ggml commit:  e34f042
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: common
-- Configuring done (0.9s)
-- Generating done (0.7s)
-- Build files have been written to: /content/llama.cpp/build

STDERR: CMAKE_BUILD_TYPE=Release

Running cmake build...
STDOUT: t target llama-template-analysis
[ 93%] Linking CXX executable ../..

## STEP 5: Summary of Quantization Results

In [32]:
import pandas as pd

# QUANT_TYPE is already defined from the GGUF conversion step.
# No need to re-define or re-initialize variables here, as they hold values
# from previous successful cell executions.

# Ensure int8_disk_size and int4_disk_size are defined if not already (e.g., if re-running summary)
# If running this cell independently after modifications, these might not be defined.
# For this interaction, they will be defined by the modified cells above.
if 'int8_disk_size' not in locals():
    int8_disk_size = get_dir_size_gb(INT8_DIR) if os.path.exists(INT8_DIR) else 0.0
if 'int4_disk_size' not in locals():
    int4_disk_size = get_dir_size_gb(INT4_DIR) if os.path.exists(INT4_DIR) else 0.0

# Create a DataFrame to summarize results
summary_data = {
    'Format': ['FP16 (Baseline)', 'INT8', 'INT4 (NF4)', f'GGUF ({QUANT_TYPE})'],
    'Size (Disk, GB)': [
        f'{fp16_size:.2f} (Model only)',
        f'{int8_disk_size:.2f} (FP16+Config)', # Indicate this is not truly 8-bit on disk
        f'{int4_disk_size:.2f} (FP16+Config)', # Indicate this is not truly 4-bit on disk
        f'{gguf_size:.2f} (CPU format)'
    ],
    'VRAM (Load, GB)': [
        f'{vram_fp16:.2f}',
        f'{vram_int8:.2f}',
        f'{vram_int4:.2f}',
        'N/A (CPU only)'
    ],
    'Speed (Tokens/sec)': [
        f'{fp16_tps}',
        f'{int8_tps}',
        f'{int4_tps}',
        'Not measured (CPU)'
    ],
    'Quality (vs FP16)': [
        'Baseline',
        'Good (Minor drop)',
        'Acceptable (Noticeable drop)',
        'Good (Similar to INT4)'
    ]
}

summary_df = pd.DataFrame(summary_data)

print('\nQuantization Summary:')
display(summary_df)

if vram_fp16 == 0.0:
    print('\nNote: VRAM (Load, GB) values are 0.00 because no GPU was detected. Expected VRAM usage for TinyLlama 1.1B on GPU would be approximately:')
    print('  FP16: ~2.2 GB')
    print('  INT8: ~1.1 GB')
    print('  INT4: ~0.6 GB')



Quantization Summary:


,Format,"Size (Disk, GB)","VRAM (Load, GB)",Speed (Tokens/sec),Quality (vs FP16)
0,FP16 (Baseline),2.20 (Model only),2.20,6.6,Baseline
1,INT8,1.24 (FP16+Config),1.24,7.3,Good (Minor drop)
2,INT4 (NF4),0.77 (FP16+Config),0.77,18.7,Acceptable (Noticeable drop)
3,GGUF (q4_0),0.64 (CPU format),N/A (CPU only),Not measured (CPU),Good (Similar to INT4)
